# AgentCore Gateway — Connect Agents to Tools at Scale

This notebook demonstrates:
- Setting up an AgentCore Gateway
- Converting a Lambda function into an MCP-compatible tool
- Registering an OpenAPI spec as agent tools
- Semantic tool discovery (agent finds the right tool by context)
- Authentication configuration (ingress + egress)

## ⚠️ Cost Warning
- AgentCore Gateway: Pay per request routed
- Lambda function: Minimal cost (few invocations)
- Model invocations: Standard Bedrock pricing
- Estimated cost for this lab: **< $1.00**
- **Cleanup**: Delete gateway and Lambda after the lab

In [21]:
# Install required packages
!pip install boto3 strands-agents strands-agents-tools bedrock-agentcore -q

In [22]:
import boto3
import json
import time

REGION = "us-west-2"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

agentcore_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)

print(f"Region: {REGION}")
print(f"Account: {ACCOUNT_ID}")

Region: us-west-2
Account: 058264544288


## 1. Create a Lambda Function (Tool Backend)

We'll create a simple Lambda that acts as a weather lookup service — this will become an agent tool via Gateway.

In [23]:
# Lambda function code
LAMBDA_CODE = '''
import json

WEATHER_DATA = {
    "seattle": {"temp": 55, "condition": "rainy", "humidity": 85},
    "miami": {"temp": 82, "condition": "sunny", "humidity": 70},
    "denver": {"temp": 45, "condition": "cloudy", "humidity": 40},
    "new york": {"temp": 62, "condition": "partly cloudy", "humidity": 55},
}

def lambda_handler(event, context):
    city = event.get("city", "").lower()
    if city in WEATHER_DATA:
        return {
            "statusCode": 200,
            "body": json.dumps({"city": city, **WEATHER_DATA[city]})
        }
    return {
        "statusCode": 404,
        "body": json.dumps({"error": f"No weather data for {city}"})
    }
'''

print("Lambda function code defined (weather lookup service)")

Lambda function code defined (weather lookup service)


In [24]:
import zipfile
import io

# Package Lambda code
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', LAMBDA_CODE)
zip_buffer.seek(0)

LAMBDA_NAME = "agentcore-gateway-lab-weather"
LAMBDA_ROLE = f"arn:aws:iam::{ACCOUNT_ID}:role/lambda-basic-execution"

# Create the role if it doesn't exist
iam_client = boto3.client("iam")
try:
    iam_client.create_role(
        RoleName="lambda-basic-execution",
        AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]})
    )
    iam_client.attach_role_policy(RoleName="lambda-basic-execution", PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    import time; time.sleep(10)
    print("✅ Created lambda-basic-execution role")
except iam_client.exceptions.EntityAlreadyExistsException:
    pass

try:
    lambda_client.create_function(
        FunctionName=LAMBDA_NAME,
        Runtime="python3.12",
        Role=LAMBDA_ROLE,
        Handler="lambda_function.lambda_handler",
        Code={"ZipFile": zip_buffer.read()},
        Timeout=30
    )
    print(f"✅ Lambda function '{LAMBDA_NAME}' created")
except lambda_client.exceptions.ResourceConflictException:
    print(f"ℹ️ Lambda function '{LAMBDA_NAME}' already exists")
except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Ensure you have a 'lambda-basic-execution' IAM role or update LAMBDA_ROLE")

ℹ️ Lambda function 'agentcore-gateway-lab-weather' already exists


## 2. Create an AgentCore Gateway

The Gateway converts your Lambda into an MCP-compatible tool that any agent framework can use.

In [25]:
# Create a Gateway
GATEWAY_NAME = "lab-weather-gateway"
GATEWAY_ROLE_NAME = "agentcore-gateway-lab-role"

# Create gateway IAM role
try:
    iam_client.create_role(
        RoleName=GATEWAY_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]})
    )
    iam_client.put_role_policy(RoleName=GATEWAY_ROLE_NAME, PolicyName="LambdaInvoke",
        PolicyDocument=json.dumps({"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"}]}))
    print(f"✅ Gateway role created")
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"ℹ️ Gateway role exists")

gateway_role_arn = f"arn:aws:iam::{ACCOUNT_ID}:role/{GATEWAY_ROLE_NAME}"

try:
    gateway_response = agentcore_client.create_gateway(
        name=GATEWAY_NAME,
        description="Gateway for weather tool lab",
        roleArn=gateway_role_arn,
        protocolType="MCP",
        authorizerType="NONE"
    )
    gateway_id = gateway_response["gatewayId"]
    gateway_arn = gateway_response["gatewayArn"]
    print(f"✅ Gateway created: {gateway_id}")
    print(f"   ARN: {gateway_arn}")
except agentcore_client.exceptions.ConflictException:
    gateways = agentcore_client.list_gateways()
    for gw in gateways.get("items", []):
        if GATEWAY_NAME in gw.get("name", ""):
            gateway_id = gw["gatewayId"]
            gateway_arn = gw.get("gatewayArn", f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{gw['gatewayId']}")
            print(f"ℹ️ Reusing existing gateway: {gateway_id}")
            break
except Exception as e:
    print(f"Error: {e}")

ℹ️ Gateway role exists
ℹ️ Reusing existing gateway: lab-weather-gateway-ej9ib7uf0z


## 3. Register Lambda as a Tool via Gateway

Gateway transforms the Lambda into an MCP tool with a schema that agents can discover and invoke.

In [26]:
# Register the Lambda as a tool
LAMBDA_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{LAMBDA_NAME}"

try:
    tool_response = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="get-weather",
        description="Get current weather conditions for a city",
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
        targetConfiguration={"mcp": {"lambda": {
            "lambdaArn": LAMBDA_ARN,
            "toolSchema": {"inlinePayload": [{
                "name": "get_weather",
                "description": "Get current weather conditions for a city",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "The city to get weather for (e.g., seattle, miami, denver)"}
                    },
                    "required": ["city"]
                }
            }]}
        }}}
    )
    print(f"✅ Tool 'get_weather' registered on gateway")
    print(f"   Lambda ARN: {LAMBDA_ARN}")
except Exception as e:
    print(f"Error: {e}")

Error: An error occurred (ConflictException) when calling the CreateGatewayTarget operation: A target with name 'get-weather' already exists in this gateway


## 4. Register an OpenAPI Spec as Tools

Gateway can also ingest an OpenAPI specification and automatically create tools for each endpoint.

In [27]:
# OpenAPI spec for a mock order management API
OPENAPI_SPEC = {
    "openapi": "3.0.0",
    "info": {"title": "Order API", "version": "1.0"},
    "paths": {
        "/orders/{orderId}": {
            "get": {
                "operationId": "getOrder",
                "summary": "Get order details by ID",
                "parameters": [
                    {"name": "orderId", "in": "path", "required": True, "schema": {"type": "string"}}
                ]
            }
        },
        "/orders": {
            "post": {
                "operationId": "createOrder",
                "summary": "Create a new order",
                "requestBody": {
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "product": {"type": "string"},
                                    "quantity": {"type": "integer"}
                                }
                            }
                        }
                    }
                }
            }
        }
    }
}

try:
    openapi_response = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="order-api",
        description="Order management API",
        targetConfiguration={"mcp": {"openApiSchema": {
            "inlinePayload": json.dumps(OPENAPI_SPEC)
        }}}
    )
    print(f"✅ OpenAPI tools registered (getOrder, createOrder)")
except Exception as e:
    print(f"ℹ️ OpenAPI target: {e}")
    print("Note: OpenAPI targets require a reachable endpoint and valid credential provider (OAuth or API Key).")
    print("Skipping for this lab — Lambda-based targets demonstrated above.")

ℹ️ OpenAPI target: An error occurred (ValidationException) when calling the CreateGatewayTarget operation: Credential provider configurations is not defined
Note: OpenAPI targets require a reachable endpoint and valid credential provider (OAuth or API Key).
Skipping for this lab — Lambda-based targets demonstrated above.


## 5. Semantic Tool Discovery

When you have many tools, agents can search for the right one by describing what they need. Gateway returns the most relevant tools.

In [28]:
# Semantic search for tools
try:
    search_response = agentcore_client.search_gateway_tools(
        gatewayId=gateway_id,
        query="I need to check the weather forecast",
        maxResults=3
    )
    print("Semantic search results for: 'I need to check the weather forecast'")
    print("=" * 60)
    for tool_result in search_response.get("tools", []):
        print(f"  Tool: {tool_result['name']}")
        print(f"  Description: {tool_result['description']}")
        print(f"  Relevance: {tool_result.get('score', 'N/A')}")
        print()
except Exception as e:
    print(f"ℹ️ Semantic search: {e}")
    print("Note: Semantic search requires tools to be indexed (may take a moment)")

ℹ️ Semantic search: 'BedrockAgentCoreControl' object has no attribute 'search_gateway_tools'
Note: Semantic search requires tools to be indexed (may take a moment)


## 6. Use Gateway Tools with a Strands Agent

Connect the Gateway endpoint to a Strands agent so it can discover and invoke tools dynamically.

In [29]:
from strands import Agent
from strands.models import BedrockModel

# Create agent with Gateway as MCP endpoint
model = BedrockModel(model_id="us.amazon.nova-pro-v1:0", region_name=REGION)

# Note: In production, you'd connect via the Gateway MCP endpoint
# For this demo, we use the tools directly
from strands import tool as strands_tool

@strands_tool
def get_weather(city: str) -> str:
    """Get current weather for a city via Gateway.
    
    Args:
        city: The city name to look up weather for
    """
    # In production: this would route through Gateway
    # Gateway handles auth, logging, policy enforcement
    response = lambda_client.invoke(
        FunctionName=LAMBDA_NAME,
        Payload=json.dumps({"city": city})
    )
    result = json.loads(response["Payload"].read())
    return result.get("body", str(result))

agent = Agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful weather assistant. Use the get_weather tool to answer questions."
)

response = agent("What's the weather like in Seattle and Miami?")
print(response)

<thinking> To find out the current weather in Seattle and Miami, I need to use the get_weather tool for both cities. </thinking>

Tool #1: get_weather

Tool #2: get_weather
The current weather in Seattle is 55 degrees Fahrenheit, and it's rainy with a humidity of 85%. In Miami, the current weather is 82 degrees Fahrenheit, and it's sunny with a humidity of 70%.The current weather in Seattle is 55 degrees Fahrenheit, and it's rainy with a humidity of 85%. In Miami, the current weather is 82 degrees Fahrenheit, and it's sunny with a humidity of 70%.



## Key Takeaways

| Feature | Benefit |
|---------|--------|
| Lambda → MCP | Convert any Lambda into an agent tool with one API call |
| OpenAPI → MCP | Auto-generate tools from existing API specs |
| Semantic discovery | Agents find the right tool from thousands without listing all in prompt |
| Auth management | Gateway handles OAuth, token refresh, credential injection |
| Framework agnostic | Works with Strands, LangGraph, CrewAI, any MCP client |

## 🧹 Cleanup (Optional)

Uncomment and run to delete resources.

In [30]:
# # Cleanup
# try:
#     agentcore_client.delete_gateway(gatewayId=gateway_id)
#     print(f"✅ Gateway {gateway_id} deleted")
# except Exception as e:
#     print(f"Gateway cleanup: {e}")
#
# try:
#     lambda_client.delete_function(FunctionName=LAMBDA_NAME)
#     print(f"✅ Lambda '{LAMBDA_NAME}' deleted")
# except Exception as e:
#     print(f"Lambda cleanup: {e}")